In [ ]:
import geopandas as gpd
import pandas as pd
from google.colab import drive

Mount Google Colab

In [ ]:
drive.mount('/content/drive')

File Paths

In [ ]:
missions_path = "/content/drive/MyDrive/MYPROJECTFOLDER/bj_missions.shp"
districts_path = "/content/drive/MyDrive/MYPROJECTFOLDER/Zim_Fixed_Constituencies.shp"

Load Data

In [ ]:
missions = gpd.read_file(missions_path)
districts = gpd.read_file(districts_path)

print(type(missions))
print(type(districts))

print(missions.columns)
print(districts.columns)

Safeguards

In [ ]:
assert isinstance(missions, gpd.GeoDataFrame)
assert isinstance(districts, gpd.GeoDataFrame)

Check Geometry Types

In [ ]:
print("Mission geometry types:")
print(missions.geom_type.unique())

print("District geometry types:")
print(districts.geom_type.unique())

Keep Only Missions in Zimbabwe

In [ ]:
missions = missions.to_crs(districts.crs)

Reproject to Common CRS

In [ ]:
missions = missions.to_crs("ESRI:102022")
districts = districts.to_crs("ESRI:102022")

Fix Geometries

In [ ]:
missions = missions[missions.is_valid]

districts["geometry"] = districts.buffer(0)

Spatial Join (Assign Each Mission to an Electoral District)

In [ ]:
missions_joined = gpd.sjoin(
    missions,
    districts,
    how="inner",
    predicate="within"
)

Count Missions per District

In [ ]:
missions_by_district = (
    missions_joined
    .groupby("Cons_name")
    .size()
    .reset_index(name="mission_count")
)

print(missions_by_district.head())

Merge Counts Back to Districts

In [ ]:
districts = districts.merge(
    missions_by_district,
    on="Cons_name",
    how="left"
)

Fill Districts with No Missions

In [ ]:
districts["mission_count"] = (
    districts["mission_count"]
    .fillna(0)
    .astype(int)
)

Export JSON

In [ ]:
output_json = (
    "/content/drive/MyDrive/MYPROJECTFOLDER/mission_counts.json"
)

df = districts.drop(columns="geometry").fillna(0)

df.to_json(output_json, orient="records")

Export GeoJSON

In [ ]:
output_geojson = (
    "/content/drive/MyDrive/MYPROJECTFOLDER/mission_counts.geojson"
)

districts.to_file(
    output_geojson,
    driver="GeoJSON"
)

print("Done.")